# CMS DoubleElectron Plot Notebook

Notebook front end for `scripts/plot.py`, based on the plotting flow in `experiments/cms_doubleelectron/cms_doubleelectron_plot_with_residual_panel.ipynb`.

It keeps the reference notebook's paper-style ratio and relative residual checks, but routes loading, model construction, device selection, caching, and output writing through the current reusable script layer.

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display


def find_repo_root(start: Path) -> Path:
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "configs" / "cms_doubleelectron_mps.yaml").exists() and (path / "scripts" / "plot.py").exists():
            return path
    raise RuntimeError("Could not find repo root containing configs/cms_doubleelectron_mps.yaml and scripts/plot.py")


REPO_ROOT = find_repo_root(Path.cwd())
PLOT_SCRIPT = REPO_ROOT / "scripts" / "plot.py"
REFERENCE_NOTEBOOK = REPO_ROOT / "experiments" / "cms_doubleelectron" / "cms_doubleelectron_plot_with_residual_panel.ipynb"

print("repo root:", REPO_ROOT)
print("plot script:", PLOT_SCRIPT)
print("reference notebook:", REFERENCE_NOTEBOOK)
print("available logical CPU threads:", os.cpu_count())

## Settings

Edit these values for the run you want to inspect. The defaults use the current root-level CMS config and the newest checked-in example run directory if it exists.

In [ ]:
CONFIG = REPO_ROOT / "configs" / "cms_doubleelectron_mps.yaml"

candidate_checkpoints = [
    REPO_ROOT / "outputs" / "cms_doubleelectron" / "v5_mps" / "best_model.pt",
    REPO_ROOT / "outputs" / "cms_doubleelectron" / "v4_mps" / "best_model.pt",
    REPO_ROOT / "outputs" / "cms_doubleelectron" / "v3_fullproduction_mps" / "best_model.pt",
]
CHECKPOINT = next((path for path in candidate_checkpoints if path.exists()), candidate_checkpoints[0])

# Keep None to use scripts/plot.py's default: <checkpoint_parent>/plots/.
OUTPUT_DIR_OVERRIDE = None
OUTPUT_DIR = CHECKPOINT.parent / "plots" if OUTPUT_DIR_OVERRIDE is None else Path(OUTPUT_DIR_OVERRIDE)
DEVICE = "auto"
SPLIT = "all"  # all, val, or test

# Use None for the full selected data. Set a smaller value for a quick notebook smoke run.
NUM_SAMPLES = None
MAX_X_EVENTS = 5_000_000
MAX_Z_EVENTS = 5_000_000
BATCH_SIZE = None
PROGRESS_LOG_STEPS = 10
THREADS = "auto"  # auto/max/all uses all logical CPUs; set an integer to cap it.
INTEROP_THREADS = "auto"
SEED = None

MASS_LOW = 70.0
MASS_HIGH = 110.0
MASS_BIN_WIDTH = 1.0
COUNTS = False
NO_DATA_CACHE = False

print("config:", CONFIG)
print("checkpoint:", CHECKPOINT)
print("output dir:", OUTPUT_DIR)

In [ ]:
def add_optional_arg(cmd: list[str], flag: str, value) -> None:
    if value is not None:
        cmd.extend([flag, str(value)])


cmd = [
    sys.executable,
    str(PLOT_SCRIPT),
    "--config", str(CONFIG),
    "--checkpoint", str(CHECKPOINT),
    "--device", DEVICE,
    "--split", SPLIT,
    "--max-x-events", str(MAX_X_EVENTS),
    "--max-z-events", str(MAX_Z_EVENTS),
    "--progress-log-steps", str(PROGRESS_LOG_STEPS),
    "--threads", str(THREADS),
    "--interop-threads", str(INTEROP_THREADS),
    "--mass-low", str(MASS_LOW),
    "--mass-high", str(MASS_HIGH),
    "--mass-bin-width", str(MASS_BIN_WIDTH),
]
add_optional_arg(cmd, "--output-dir", OUTPUT_DIR_OVERRIDE)
add_optional_arg(cmd, "--num-samples", NUM_SAMPLES)
add_optional_arg(cmd, "--batch-size", BATCH_SIZE)
add_optional_arg(cmd, "--seed", SEED)
if COUNTS:
    cmd.append("--counts")
if NO_DATA_CACHE:
    cmd.append("--no-data-cache")

display(Markdown("```bash\n" + shlex.join(cmd) + "\n```"))

In [ ]:
result = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
result.check_returncode()

## Generated Artifacts

In [ ]:
expected_plots = [
    "paperstyle_xspace_mass_density_ratio.png",
    "paperstyle_xspace_pt_density_ratio.png",
    "paperstyle_zspace_mass_density_ratio.png",
    "paperstyle_xspace_pos_py_ratio.png",
    "paperstyle_xspace_pos_pz_ratio.png",
    "paperstyle_xspace_pos_E_ratio.png",
    "paperstyle_zspace_pos_py_ratio.png",
    "paperstyle_zspace_pos_pz_ratio.png",
    "paperstyle_zspace_pos_E_ratio.png",
    "all8_xspace_components_density.png",
    "all8_zspace_components_density.png",
    "zspace_validation/zspace_mass_mg5_vs_x2z.png",
    "zspace_validation/zspace_components_mg5_vs_x2z.png",
]

for name in expected_plots:
    path = OUTPUT_DIR / name
    print(("OK  " if path.exists() else "MISS"), path)

In [ ]:
for name in expected_plots[:9]:
    path = OUTPUT_DIR / name
    if path.exists():
        display(Markdown(f"### `{name}`"))
        display(Image(filename=str(path)))

In [ ]:
summary_txt = OUTPUT_DIR / "paperstyle_summary.txt"
summary_json = OUTPUT_DIR / "paperstyle_summary.json"

if summary_txt.exists():
    display(Markdown("### `paperstyle_summary.txt`"))
    print(summary_txt.read_text(encoding="utf-8"))

if summary_json.exists():
    summary = json.loads(summary_json.read_text(encoding="utf-8"))
    display(Markdown("### Key residual metrics from `paperstyle_summary.json`"))
    for key, value in summary.get("observable_residuals", {}).items():
        print(
            key,
            "valid_bins=", value.get("valid_bins"),
            "rms=", value.get("rms_rel_residual"),
            "mae=", value.get("mae_rel_residual"),
            "max_abs=", value.get("max_abs_rel_residual"),
        )

## Notes

- The reference notebook is `experiments/cms_doubleelectron/cms_doubleelectron_plot_with_residual_panel.ipynb`.
- This notebook intentionally delegates the plotting implementation to `scripts/plot.py`, so notebook and CLI runs share the same data/model path.
- `paperstyle_loaded_model_outputs.npz` contains the arrays used for the displayed plots if deeper interactive checks are needed.